# H04：14:30 日频与 L2 融合技术验收

本 Notebook 保存固定 H03 基线上的 H04 实际运行过程和结果；研究状态、最终结论及采用决定
只由同目录 `README.md` 的 H04 段落拥有。这里不评价因子预测价值。

输入：四组 P/T 的 daily P 与 L2 T、2025/2026 正式 calendar 及其 Meta 直接引用的
两个既有 raw calendar 对象，共 24 个 Meta/payload 文件。
开发样本为 2025-11-18，其余三组为预注册的最终技术验证。
计算口径与停止条件已在执行前写入 README；任一断言失败保留现场并停止。

核心问题是精确一致性和失败边界，因此使用小型结果表格，不生成预测效果图。

2026-09-13 保存运行 `validation-ygivxka9`：四组共 20,698 行，28 个排名列与独立重算精确一致；
8 次新建、8 次复用及 2 次预设失败检查均符合预期。具体状态和结论见 README 的 H04。

## 参数与可恢复基线

使用项目 uv 锁定的 Python 内核执行。代码从本轮 `source/` 快照以独立 CLI 进程运行，
输入只来自本轮 `inputs/` 归档；每次执行创建新的隔离输出目录，不写权威数据。
快照 `.env.dev` 只含无效占位凭证，子进程环境显式构造，不继承生产凭证。

复跑前保持以下 evidence 路径可用，并用 `uv sync --locked --extra test` 准备项目环境。
`build_notebook.py` 使用独立的 nbformat 5.11.1 / nbclient 0.11.0 authoring 环境，
实际执行内核和 CLI 均使用下方锁定解释器。

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

EVIDENCE = Path('/home/wsw/app/research-evidence/stock-1430-h04-2026-09-13-gcp_qa0b')
SOURCE = EVIDENCE / 'source'
INPUTS = EVIDENCE / 'inputs'
LOCKED_PYTHON = Path('/home/wsw/app/dev/trading/.venv/bin/python')
BASELINE = '62078a4ed313ea97b2d332ef564d5fa921352e66'
PAIRS = (
    ('2025-11-17', '2025-11-18'),
    ('2025-12-31', '2026-01-05'),
    ('2026-04-30', '2026-05-06'),
    ('2026-07-24', '2026-07-27'),
)
DAILY_COLUMNS = (
    'f_d_close_return_1d', 'f_d_open_gap_1d', 'f_d_log_amount',
    'f_d_max_drawdown_20d_asof_tminus1',
    'f_d_close_volatility_60d_asof_tminus1',
    'f_d_close_return_5d_asof_tminus1',
    'f_d_turnover_rate_mean_20d_asof_tminus1',
)
KEY_COLUMNS = ('symbol', 'trade_date', 'decision_ts_utc')
assert Path(sys.executable) == LOCKED_PYTHON
assert sys.version_info[:2] == (3, 13)
versions = {name: metadata.version(name) for name in ('numpy', 'pandas', 'pyarrow')}
assert versions == {'numpy': '2.5.1', 'pandas': '3.0.2', 'pyarrow': '25.0.0'}
RUN = Path(tempfile.mkdtemp(prefix='validation-', dir=EVIDENCE))
environment = {'python': sys.version, 'python_executable': sys.executable, 'packages': versions,
               'baseline': BASELINE, 'randomness': 'none', 'pairs': PAIRS}
(RUN / 'environment.json').write_text(json.dumps(environment, indent=2) + '\n')
print('输出目录：', RUN)
print('运行环境：', versions)

输出目录： /home/wsw/app/research-evidence/stock-1430-h04-2026-09-13-gcp_qa0b/validation-ygivxka9
运行环境： {'numpy': '2.5.1', 'pandas': '3.0.2', 'pyarrow': '25.0.0'}


## 冻结输入与源码

内容摘要与可恢复内容同时保留。版本名本身不证明历史文件未修订；本轮验证输入选择与
计算截断，不声称历史实时就绪或完成上游 point-in-time 审计。

In [2]:
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

input_manifest = json.loads((EVIDENCE / 'input_manifest.json').read_text())
assert digest(EVIDENCE / 'input_manifest.json') == '576b69eb44b303ba3b4bec2dac070e6fcc12b3b1ffe893d64c91e4b044b0e7ef'
runtime_manifest = json.loads((EVIDENCE / 'runtime_manifest.json').read_text())
for entry in input_manifest:
    assert digest(INPUTS / entry['path']) == entry['sha256'], entry['path']
for entry in runtime_manifest:
    assert digest(SOURCE / entry['path']) == entry['sha256'], entry['path']

def read_table(path):
    with pq.ParquetFile(path) as reader:
        return reader.read()

def feature_path(root, name, day):
    return root / 'features' / name / 'v1' / f'trade_date={day}' / 'data.parquet'

calendar = pd.concat([
    read_table(INPUTS / 'processed/trade_calendar/v1' / f'year={year}' / 'data.parquet').to_pandas()
    for year in (2025, 2026)
], ignore_index=True)
sessions = sorted(calendar.loc[calendar['is_open'].eq(True), 'trade_date'])
for previous, target in PAIRS:
    assert sessions[sessions.index(target) - 1] == previous
print(f'{len(input_manifest)} 个输入文件、{len(runtime_manifest)} 个运行源码/测试/依赖文件摘要通过。')
print('四组 P/T 均由归档正式 calendar 确认。')

24 个输入文件、232 个运行源码/测试/依赖文件摘要通过。
四组 P/T 均由归档正式 calendar 确认。


## 从两个全新存储根构建

第一份存储没有 daily T；第二份存储故意放入不可读取的 daily T Meta。
每个目标通过真实 CLI 构建，逐次保留 stdout、stderr、退出码、wall time 与 peak RSS。

In [3]:
commands = []
def run_cli(storage, previous, target, label, expected_exit=0):
    output = RUN / 'commands' / label
    output.mkdir(parents=True)
    argv = ['/usr/bin/time', '-f', '%e %M', '-o', str(output / 'resources.txt'),
            str(LOCKED_PYTHON), '-B', '-m', 'src.cli', 'data-stock-1430-fusion-backfill',
            '--start', target, '--end', target]
    env = {'PATH': str(LOCKED_PYTHON.parent) + ':/usr/bin:/bin', 'LANG': 'C.UTF-8',
           'PYTHONDONTWRITEBYTECODE': '1', 'ENV': 'dev', 'ZERO_STORAGE_ROOT': str(storage)}
    start = time.perf_counter()
    result = subprocess.run(argv, cwd=SOURCE, env=env, capture_output=True, text=True)
    elapsed = time.perf_counter() - start
    (output / 'stdout.txt').write_text(result.stdout)
    (output / 'stderr.txt').write_text(result.stderr)
    resource_lines = (output / 'resources.txt').read_text().strip().splitlines()
    measured_seconds, peak_rss_kib = resource_lines[-1].split()
    record = {'P': previous, 'T': target, 'label': label, 'argv': argv, 'env': env,
              'cwd': str(SOURCE), 'exit_code': result.returncode, 'expected_exit': expected_exit,
              'wall_seconds': elapsed, 'time_wall_seconds': float(measured_seconds),
              'peak_rss_kib': int(peak_rss_kib)}
    (output / 'command.json').write_text(json.dumps(record, indent=2) + '\n')
    commands.append(record)
    assert result.returncode == expected_exit, f'{label}: {result.stderr[-3000:]}'
    return record

first_root, second_root = RUN / 'first', RUN / 'second'
for root in (first_root, second_root):
    shutil.copytree(INPUTS, root)
for _, target in PAIRS:
    invalid_daily_t = feature_path(second_root, 'tushare_daily_basic', target).with_name('meta.json')
    invalid_daily_t.parent.mkdir(parents=True)
    invalid_daily_t.write_text('invalid JSON; future daily must never be read')
for previous, target in PAIRS:
    run_cli(first_root, previous, target, f'first-{target}')
    run_cli(second_root, previous, target, f'second-{target}')
print('8 次首次构建均返回 0，daily T 不影响构建。')

8 次首次构建均返回 0，daily T 不影响构建。


## 独立重算、key 与 coverage

用 NumPy 排序与左右插入位置独立计算平均序位，逐值精确比较生产 builder 的 Pandas rank。
分母为每个目标 universe 内该列的有效数量。完整行比例使用全部 39 列，不删除输出行。

In [4]:
rows = []
coverage = []
for previous, target in PAIRS:
    l2 = read_table(feature_path(INPUTS, 'l2_stock_1430', target))
    daily = read_table(feature_path(INPUTS, 'tushare_daily_basic', previous))
    output = read_table(feature_path(first_root, 'stock_1430_daily_l2', target))
    rerun = read_table(feature_path(second_root, 'stock_1430_daily_l2', target))
    assert output.equals(rerun, check_metadata=True)
    assert output.num_columns == 42 and output.num_rows == l2.num_rows > 0
    assert output.select(l2.column_names).equals(l2)
    assert output.column_names[35:] == [f'{column}_rank' for column in DAILY_COLUMNS]
    expected_schema = pa.schema([*l2.schema, *[pa.field(f'{column}_rank', pa.float64()) for column in DAILY_COLUMNS]])
    assert output.schema.equals(expected_schema, check_metadata=False)
    symbols = l2.column('symbol').to_pylist()
    assert len(symbols) == len(set(symbols))
    assert symbols == sorted(symbols)
    assert set(output.column('trade_date').to_pylist()) == {target}
    decision = pd.Timestamp(target + ' 14:30:00', tz='Asia/Shanghai').value // 1000
    assert set(output.column('decision_ts_utc').to_pylist()) == {decision}
    daily_symbols = daily.column('symbol').to_pylist()
    for column in DAILY_COLUMNS:
        by_symbol = dict(zip(daily_symbols, daily.column(column).to_pylist(), strict=True))
        aligned = np.array([by_symbol.get(symbol) for symbol in symbols], dtype='float64')
        valid = np.isfinite(aligned)
        valid_count = int(valid.sum())
        assert valid_count > 0
        ordered = np.sort(aligned[valid])
        expected = np.full(len(aligned), np.nan)
        left = np.searchsorted(ordered, aligned[valid], side='left')
        right = np.searchsorted(ordered, aligned[valid], side='right')
        expected[valid] = (left + right + 1) / (2 * valid_count)
        actual_column = output.column(f'{column}_rank')
        actual = actual_column.to_numpy(zero_copy_only=False)
        assert np.array_equal(actual, expected, equal_nan=True), (target, column)
        assert actual_column.null_count == len(symbols) - valid_count
        coverage.append({'T': target, 'column': column, 'valid': valid_count,
                         'rows': len(symbols), 'coverage': valid_count / len(symbols)})
    complete = np.column_stack([
        np.isfinite(output.column(column).to_numpy(zero_copy_only=False))
        for column in output.column_names[3:]
    ]).all(axis=1)
    keys = list(zip(*(output.column(column).to_pylist() for column in KEY_COLUMNS), strict=True))
    schema = [(field.name, str(field.type), field.nullable) for field in output.schema]
    key_digest = hashlib.sha256(json.dumps(keys, separators=(',', ':')).encode()).hexdigest()
    schema_digest = hashlib.sha256(json.dumps(schema, separators=(',', ':')).encode()).hexdigest()
    first_command = next(command for command in commands if command['label'] == f'first-{target}')
    rows.append({'P': previous, 'T': target, 'rows': len(symbols),
                 'daily_matched': len(set(symbols) & set(daily_symbols)),
                 'complete_39': int(complete.sum()), 'key_sha256': key_digest,
                 'schema_sha256': schema_digest, 'wall_seconds': first_command['wall_seconds'],
                 'peak_rss_kib': first_command['peak_rss_kib']})
display(pd.DataFrame(rows)[['P', 'T', 'rows', 'daily_matched', 'complete_39', 'wall_seconds', 'peak_rss_kib']].round({'wall_seconds': 3}))
print('8 个新建分区逐字段一致，四组共 28 个排名列与独立重算精确一致。')

,P,T,rows,daily_matched,complete_39,wall_seconds,peak_rss_kib
0,2025-11-17,2025-11-18,5157,5153,4845,1.178,325484
1,2025-12-31,2026-01-05,5170,5166,4841,1.149,321644
2,2026-04-30,2026-05-06,5179,5144,4839,1.181,321636
3,2026-07-24,2026-07-27,5192,5191,4673,1.185,325892


8 个新建分区逐字段一致，四组共 28 个排名列与独立重算精确一致。


## 七列有效值覆盖率

以下百分比使用各日完整 L2 universe 为分母，字段名指 P 分区源列。

In [5]:
coverage_frame = pd.DataFrame(coverage)
display(coverage_frame.pivot(index='column', columns='T', values='coverage').reindex(DAILY_COLUMNS).mul(100).round(3).rename_axis('有效值占 L2 universe 的百分比'))
print('本表只描述本轮样本的覆盖率，不是因子效果或全历史质量结论。')

T,2025-11-18,2026-01-05,2026-05-06,2026-07-27
有效值占 L2 universe 的百分比,,,,
f_d_close_return_1d,99.903,99.865,98.533,99.981
f_d_open_gap_1d,99.903,99.865,98.533,99.981
f_d_log_amount,99.922,99.923,99.324,99.981
f_d_max_drawdown_20d_asof_tminus1,99.186,98.646,97.026,99.307
f_d_close_volatility_60d_asof_tminus1,97.886,97.118,95.829,95.898
f_d_close_return_5d_asof_tminus1,99.729,99.458,97.799,99.846
f_d_turnover_rate_mean_20d_asof_tminus1,99.186,98.646,97.026,99.210


本表只描述本轮样本的覆盖率，不是因子效果或全历史质量结论。


## 复用与失败现场

首轮输出在上游移走后复用；第二轮输出在上游内容损坏后复用。比较输出 SHA、size、mtime
和 inode。另在新根验证缺失 daily P 以及无效输出 Meta 的失败行为。

In [6]:
def output_identities(root):
    result = {}
    for _, target in PAIRS:
        payload = feature_path(root, 'stock_1430_daily_l2', target)
        for path in (payload, payload.with_name('meta.json')):
            stat = path.stat()
            result[str(path.relative_to(root))] = [digest(path), stat.st_size, stat.st_mtime_ns, stat.st_ino]
    return result

first_before, second_before = output_identities(first_root), output_identities(second_root)
for name in ('l2_stock_1430', 'tushare_daily_basic'):
    (first_root / 'features' / name).rename(first_root / f'unavailable-{name}')
for entry in input_manifest:
    if entry['path'].startswith('features/') and entry['path'].endswith('data.parquet'):
        (second_root / entry['path']).write_bytes(b'changed upstream payload')
for previous, target in PAIRS:
    run_cli(first_root, previous, target, f'missing-input-reuse-{target}')
    run_cli(second_root, previous, target, f'changed-input-reuse-{target}')
assert output_identities(first_root) == first_before
assert output_identities(second_root) == second_before

missing_root = RUN / 'missing-daily'
shutil.copytree(INPUTS, missing_root)
previous, target = PAIRS[0]
missing_payload = feature_path(missing_root, 'tushare_daily_basic', previous)
missing_payload.parent.rename(missing_root / 'withheld-daily-P')
run_cli(missing_root, previous, target, 'missing-daily-P', expected_exit=1)
output_path = feature_path(missing_root, 'stock_1430_daily_l2', target)
assert not output_path.exists() and not output_path.with_name('meta.json').exists()

invalid_root = RUN / 'invalid-output'
shutil.copytree(INPUTS, invalid_root)
invalid_payload = feature_path(invalid_root, 'stock_1430_daily_l2', target)
invalid_payload.parent.mkdir(parents=True)
invalid_payload.write_bytes(b'existing payload must stay unchanged')
invalid_meta = invalid_payload.with_name('meta.json')
invalid_meta.write_text(json.dumps({'payload': 'data.parquet', 'size_bytes': 0}))
invalid_before = [(path.read_bytes(), path.stat().st_mtime_ns) for path in (invalid_payload, invalid_meta)]
run_cli(invalid_root, previous, target, 'invalid-output-meta', expected_exit=1)
assert [(path.read_bytes(), path.stat().st_mtime_ns) for path in (invalid_payload, invalid_meta)] == invalid_before
for entry in input_manifest:
    assert digest(INPUTS / entry['path']) == entry['sha256']
print('8 次复用均返回 0，全部输出文件内容与身份不变；两个预设负例返回 1，未发布或覆盖输出。')

8 次复用均返回 0，全部输出文件内容与身份不变；两个预设负例返回 1，未发布或覆盖输出。


## 保存运行事实

结果只覆盖上述固定输入及本轮技术契约。运行时 Meta 的尺寸保证不等于内容 hash 校验；
源码、输入内容摘要与恢复副本另行保存。证据至少保留至 H04 采用/拒绝决定和对应审查结束。

In [7]:
result = {'baseline': BASELINE, 'input_manifest_sha256': digest(EVIDENCE / 'input_manifest.json'),
          'runtime_manifest_sha256': digest(EVIDENCE / 'runtime_manifest.json'),
          'rows': rows, 'coverage': coverage, 'commands': commands,
          'assertions_passed': True, 'expected_failures': 2,
          'first_output_identities': first_before, 'second_output_identities': second_before,
          'scope': 'H04 fixed-input technical validation; no alpha or historical real-time readiness claim'}
(RUN / 'result.json').write_text(json.dumps(result, indent=2) + '\n')
print('结果文件：', RUN / 'result.json')
print('新建 8 次，复用 8 次，预设失败 2 次；全部断言通过。')

结果文件： /home/wsw/app/research-evidence/stock-1430-h04-2026-09-13-gcp_qa0b/validation-ygivxka9/result.json
新建 8 次，复用 8 次，预设失败 2 次；全部断言通过。


## 保存与预览说明

7 个代码单元已从头执行，格式、执行序号和保存输出已检查。当前执行环境没有可用图形 viewer，
尚未完成视觉检查；请在浏览器打开证据目录的 `h04-validation.html`，核对四行运行汇总、
七行 coverage 与长字段名是否完整可读。代码、原始输入、输出、命令和运行环境在同一证据根保留。